#Enterprise Fleet Analytics Pipeline: Focuses on the business outcome (analytics) and the domain (fleet/logistics)

##1. Data Munging -
###1. Visibily/Manually opening the file and capture couple of data patterns (Manual Exploratory Data Analysis)
Logistics source1 file:

Format- CSV
delimiter - comma
line sep - new line
header in file - yes
missing columns
extra columns
data for format issues- id and age(expected int but received string)
empty rows

Logistics source2 file:

Format- CSV
delimiter - comma
line sep - new line
header in file - yes
missing columns
data for format issues- id and age(expected int but received string)
empty rows

###2. Programatically try to find couple of data patterns applying below EDA (File: logistics_source1)
Apply inferSchema and toDF to create a DF and analyse the actual data.
Analyse the schema, datatypes, columns etc.,
Analyse the duplicate records count and summary of the dataframe.

In [0]:
%sql
-- Create catalog only if it doesn't exist
CREATE CATALOG IF NOT EXISTS usecase_data;

-- Create schema only if it doesn't exist
CREATE SCHEMA IF NOT EXISTS usecase_data.logistics_proj_data;

-- Create volume only if it doesn't exist
CREATE VOLUME IF NOT EXISTS usecase_data.logistics_proj_data.projdata;


In [0]:
# List all volumes inside your schema
dbutils.fs.ls("dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/")

In [0]:
dbutils.fs.ls("dbfs:/Workspace/Users/viggneshwar@gmail.com/databricks/Databricks_Sample/DataFiles/Logistics/")

In [0]:
source_dir = "dbfs:/Workspace/Users/viggneshwar@gmail.com/databricks/Databricks_Sample/DataFiles/Logistics/"
dest_dir   = "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/"

# List all files in source directory
files = dbutils.fs.ls(source_dir)

for f in files:
    dest_path = dest_dir + f.name
    try:
        # Check if destination file already exists
        dbutils.fs.ls(dest_path)
        print(f"Skipping {f.name} (already exists).")
    except Exception:
        # Copy only if not exists
        dbutils.fs.cp(f.path, dest_path)
        print(f"Copied {f.name} to {dest_path}.")

In [0]:
pass_mung_df=spark.read.csv("dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source1.txt",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role")
pass_mung_df.printSchema()
#from printSchema able to understand that shipment_id and age is not in expected formats
print("Total row count is:",pass_mung_df.count())
print("The number of distinct rows:",pass_mung_df.distinct().count())

#from distinct() and dropDuplicates() able to identify the row and column level duplicates correspondingly
print("The number of distinct values in the column 'shipment_id':",pass_mung_df.dropDuplicates(["shipment_id"]).count())

#summary or describe helps us to understand statistical data understanding
#it shows there are nulls in few columns values, avg, stddev, min, max, percentile distribution
pass_mung_df.summary().show()
pass_mung_df.describe().show()

#Using columns, schema, dtypes properties to understand the column names, datatypes
print(pass_mung_df.columns)
print(pass_mung_df.schema)
print(pass_mung_df.dtypes)

###a. Passive Data Munging - (File: logistics_source1 and logistics_source2)
Without modifying the data, identify:
Shipment IDs that appear in both master_v1 and master_v2
Records where:

shipment_id is non-numeric
age is not an integer
Count rows having: 3. fewer columns than expected 4. more columns than expected

In [0]:
#Create a Spark Session Object
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("MY Spark Session").getOrCreate()
print(spark)
     

In [0]:
# Import all functions from pyspark.sql.functions (e.g., col, when, etc.)
from pyspark.sql.functions import *

# Import all data types from pyspark.sql.types (e.g., StringType, StructType, etc.)
from pyspark.sql.types import *

# -----------------------------
# Define schema for logistics_source1
# -----------------------------
schema1 = StructType([
    StructField("shipment_id", StringType(), True),   # Shipment ID as string (nullable)
    StructField("first_name", StringType(), True),    # First name of person
    StructField("last_name", StringType(), True),     # Last name of person
    StructField("age", StringType(), True),           # Age stored as string
    StructField("role", StringType(), True),          # Role of person
    StructField("corrupt_record", StringType(), True) # Column to capture malformed rows
])

# -----------------------------
# Read CSV file into DataFrame with schema
# -----------------------------
pass_mung_df = spark.read.schema(schema1) \
    .csv("dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source1.txt",
         columnNameOfCorruptRecord="corrupt_record",  # Capture bad rows in corrupt_record column
         mode="PERMISSIVE",                           # Allow malformed rows instead of failing
         header=True)                                 # First row contains column headers

# -----------------------------
# Identify rows with fewer or extra columns (corrupt records)
# -----------------------------
display(pass_mung_df.where("corrupt_record is not null"))

# -----------------------------
# Identify non-integer shipment_id values
# -----------------------------
display(
    pass_mung_df.withColumn("non_int_id", col("shipment_id").try_cast("int")) # Try casting shipment_id to int
    .select("shipment_id", "non_int_id")                                      # Select original and casted values
    .where("non_int_id is null and shipment_id is not null")                  # Keep rows where cast failed
)

# -----------------------------
# Identify non-integer age values
# -----------------------------
display(
    pass_mung_df.withColumn("non_int_age", col("age").try_cast("int"))        # Try casting age to int
    .select("shipment_id","age", "non_int_age")                               # Select shipment_id, age, and casted value
    .where("non_int_age is null and age is not null")                         # Keep rows where cast failed
)

In [0]:
# Import all functions from pyspark.sql.functions (e.g., col, when, lit, etc.)
from pyspark.sql.functions import *

# Import all data types from pyspark.sql.types (e.g., StringType, StructType, etc.)
from pyspark.sql.types import *

# -----------------------------
# Define schema for logistics_source2
# -----------------------------
schema2 = StructType([
    StructField("shipment_id", StringType(), True),   # Shipment ID as string (nullable)
    StructField("first_name", StringType(), True),    # First name of person
    StructField("last_name", StringType(), True),     # Last name of person
    StructField("age", StringType(), True),           # Age stored as string
    StructField("role", StringType(), True),          # Role of person
    StructField("hub_location", StringType(), True),  # Logistics hub location
    StructField("vehicle_type", StringType(), True),  # Vehicle type used for shipment
    StructField("corrupt_record", StringType(), True) # Column to capture malformed rows
])

# -----------------------------
# Read CSV file into DataFrame with schema
# -----------------------------
pass_mung_df2 = spark.read.schema(schema2) \
    .csv("dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source2.txt", 
         columnNameOfCorruptRecord="corrupt_record",  # Capture bad rows in corrupt_record column
         mode="PERMISSIVE",                           # Allow malformed rows instead of failing
         header=True)                                 # First row contains column headers

# -----------------------------
# Identify rows with fewer or extra columns (corrupt records)
# -----------------------------
display(pass_mung_df2.where("corrupt_record is not null"))

# -----------------------------
# Identify non-integer shipment_id values
# -----------------------------
display(
    pass_mung_df2.withColumn("non_int_id", col("shipment_id").try_cast("int")) # Try casting shipment_id to int
    .select("shipment_id", "non_int_id")                                       # Select original and casted values
    .where("non_int_id is null and shipment_id is not null")                   # Keep rows where cast failed
)

# -----------------------------
# Identify non-integer age values
# -----------------------------
display(
    pass_mung_df2.withColumn("non_int_age", col("age").try_cast("int"))        # Try casting age to int
    .select("shipment_id","age", "non_int_age")                                # Select shipment_id, age, and casted value
    .where("non_int_age is null and age is not null")                          # Keep rows where cast failed
)

# -----------------------------
# Display the full DataFrame for inspection
# -----------------------------
display(pass_mung_df2)

In [0]:
display(pass_mung_df.join(pass_mung_df2,how="inner",on='shipment_id').select("shipment_id"))
     

###b. Active Data Munging File: logistics_source1 and logistics_source2
1.Combining Data + Schema Merging (Structuring)
Read both files without enforcing schema

Align them into a single canonical schema: shipment_id, first_name, last_name, age, role, hub_location, vehicle_type, data_source
Add data_source column with values as: system1, system2 in the respective dataframes

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
active_mung_df1=spark.read.csv("dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source1.txt",inferSchema=True,header=True).withColumn("data_source",lit("system1"))
active_mung_df2=spark.read.csv("dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_source2.txt",inferSchema=True,header=True).withColumn("data_source",lit("system2")).withColumn("shipment_id", col("shipment_id").cast("string"))
print(active_mung_df1.count())
print(active_mung_df2.count())
#using unionByName to achive schema Mergeing of 2 dataframes with different columns 
active_mung_df=active_mung_df1.unionByName(active_mung_df2,allowMissingColumns=True)

display(active_mung_df)

###2. Cleansing, Scrubbing:
Cleansing (removal of unwanted datasets)

Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role
Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name

Join Readiness Rule - Drop records where the join key is null: shipment_id
Scrubbing (convert raw to tidy)

4. Age Defaulting Rule - Fill NULL values in the age column with: -1

5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN

6. Invalid Age Replacement - Replace the following values in age: "ten" to -1 "" to -1

7. Vehicle Type Normalization - Replace inconsistent vehicle types: truck to LMV bike to TwoWheeler

In [0]:
# Drop rows where either 'shipment_id' or 'role' is NULL
# Then drop rows where BOTH 'first_name' and 'last_name' are NULL
null_dropped_df = active_mung_df.na.drop(how="any", subset=['shipment_id','role']).na.drop(how="all", subset=['first_name','last_name']) 
    
# Display the cleansed DataFrame
display(null_dropped_df)

In [0]:
# Fill missing values in 'age' column with -1
# This ensures that null ages are replaced with a numeric placeholder
scrubbed_df = null_dropped_df.na.fill(-1, ['age']) \
.na.fill('UNKNOWN', ['vehicle_type']) \
.na.replace({"ten": "-1", "": "-1"}, subset=['age']) \
.na.replace({"Truck": "LMV", "Bike": "TwoWheeler"}, subset=['vehicle_type'])
    # Fill missing values in 'vehicle_type' column with 'UNKNOWN'
    # This ensures that null vehicle types are replaced with a string placeholder

    # Replace specific string values in 'age' column
    # "ten" and empty string "" are replaced with "-1" to standardize invalid entries
   
    # Replace specific string values in 'vehicle_type' column
    # "Truck" is standardized to "LMV" and "Bike" to "TwoWheeler"

# Display the cleansed and standardized DataFrame
display(scrubbed_df)

### 3. Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format
Creating shipments Details data Dataframe creation

Create a DF by Reading Data from logistics_shipment_detail.json
As this data is a clean json data, it doesn't require any cleansing or scrubbing.

In [0]:
# Read JSON file into a Spark DataFrame
# multiLine=True allows parsing JSON objects that span multiple lines
shipment_df = spark.read.json(
    "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/logistics_shipment_detail_3000.json",
    multiLine=True
)

# Show the first 5 rows of the DataFrame
shipment_df.show(5)

# Print the schema of the DataFrame to understand its structure
shipment_df.printSchema()

####Standardizations:

1. Add a column
Source File: DF of logistics_shipment_detail_3000.json
: domain as 'Logistics', current timestamp 'ingestion_timestamp' and 'False' as 'is_expedited'

2. Column Uniformity: role - Convert to lowercase
Source File: DF of merged(logistics_source1 & logistics_source2)
vehicle_type - Convert values to UPPERCASE
Source Files: DF of logistics_shipment_detail_3000.json hub_location - Convert values to initcap case
Source Files: DF of merged(logistics_source1 & logistics_source2)

3. Format Standardization:
Source Files: DF of logistics_shipment_detail_3000.json
Convert shipment_date to yyyy-MM-dd
Ensure shipment_cost has 2 decimal precision

4. Data Type Standardization
Standardizing column data types to fix schema drift and enable mathematical operations.
Source File: DF of merged(logistics_source1 & logistics_source2)
age: Cast String to Integer
Source File: DF of logistics_shipment_detail_3000.json
shipment_weight_kg: Cast to Double
Source File: DF of logistics_shipment_detail_3000.json
is_expedited: Cast to Boolean

5. Naming Standardization
Source File: DF of merged(logistics_source1 & logistics_source2)
Rename: first_name to staff_first_name
Rename: last_name to staff_last_name
Rename: hub_location to origin_hub_city

6. Reordering columns logically in a better standard format:
Source File: DF of Data from all 3 files
shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

In [0]:
# Add new columns to shipment_df
shipment_df = shipment_df \
    .withColumn("domain", lit("Logistics")) \
    .withColumn("is_expedited", lit(False)) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Column Uniformity
shipment_df = shipment_df.withColumn("vehicle_type", upper(col("vehicle_type")))
uniform_df = scrubbed_df \
    .withColumn("role", lower(col("role"))) \
    .withColumn("hub_location", initcap(col("hub_location")))

# Format Standardization
shipment_df = shipment_df \
    .withColumn("shipment_date", to_date("shipment_date", "yy-MM-dd")) \
    .withColumn("shipment_cost", round(col("shipment_cost"), 2))

# Data Type Standardisation
uniform_df = uniform_df \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("shipment_id", col("shipment_id").try_cast("long")) \
    .na.drop(how="any", subset=["shipment_id"])

shipment_df = shipment_df \
    .withColumn("shipment_weight_kg", col("shipment_weight_kg").cast("double")) \
    .withColumn("is_expedited", col("is_expedited").cast("boolean"))

# Naming Standardisation
uniform_df = uniform_df \
    .withColumnRenamed("first_name", "staff_first_name") \
    .withColumnRenamed("last_name", "staff_last_name") \
    .withColumnRenamed("hub_location", "origin_hub_city")

# Column reordering and join
logistics_std_df = uniform_df.join(shipment_df, how="inner", on="shipment_id") \
    .select("shipment_id", "staff_first_name", "staff_last_name", "role", 
            "origin_hub_city", "shipment_cost", "ingestion_timestamp")

display(logistics_std_df)

Deduplication:


Apply Record Level De-Duplication

Apply Column Level De-Duplication (Primary Key Enforcement)

In [0]:
# Remove exact duplicate rows across all columns
uniform_df = uniform_df.distinct()

# Remove duplicate rows based only on 'shipment_id'
# Keeps the first occurrence of each shipment_id and drops the rest
uniform_df = uniform_df.dropDuplicates(["shipment_id"])

# Display the cleaned DataFrame
display(uniform_df)

##2. Data Enrichment - Detailing of data
Makes your data rich and detailed

Adding of Columns (Data Enrichment)
Creating new derived attributes to enhance traceability and analytical capability.

###1. Add Audit Timestamp (load_dt) Source File: DF of logistics_source1 and logistics_source2

Scenario: We need to track exactly when this record was ingested into our Data Lakehouse for auditing purposes.
Action: Add a column load_dt using the function current_timestamp().
###2. Create Full Name (full_name) Source File: DF of logistics_source1 and logistics_source2

Scenario: The reporting dashboard requires a single field for the driver's name instead of separate columns.
Action: Create full_name by concatenating first_name and last_name with a space separator.
Result: "Rajesh" + " " + "Kumar" -> "Rajesh Kumar"
###3. Define Route Segment (route_segment) Source File: DF of logistics_shipment_detail_3000.json

Scenario: The logistics team wants to analyze performance based on specific transport lanes (Source to Destination).
Action: Combine source_city and destination_city with a hyphen.
Result: "Chennai" + "-" + "Pune" -> "Chennai-Pune"
###4. Generate Vehicle Identifier (vehicle_identifier) Source File: DF of logistics_shipment_detail_3000.json

Scenario: We need a unique tracking code that immediately tells us the vehicle type and the shipment ID.
Action: Combine vehicle_type and shipment_id to create a composite key.
Result: "Truck" + "_" + "500001" -> "Truck_500001"

In [0]:
from pyspark.sql.functions import *

# -----------------------------
# Add new columns to uniform_df
# -----------------------------
uniform_df = uniform_df.withColumn("load_dt", current_timestamp())   # Add load date/time column
uniform_df = uniform_df.withColumn("full_name",                     # Derive full name
                                   concat(col("staff_first_name"), lit(" "), col("staff_last_name")))
# Note: concat_ws could also be used to handle nulls more gracefully

# -----------------------------
# Derive new columns in shipment_df
# -----------------------------
shipment_df = shipment_df.withColumn("route_segment",                # Create route segment as "source-destination"
                                     concat(col("source_city"), lit("-"), col("destination_city")))

shipment_df = shipment_df.withColumn("vehicle_identifier",           # Create vehicle identifier as "vehicleType_shipmentId"
                                     concat(col("vehicle_type"), lit("_"), col("shipment_id")))

# -----------------------------
# Display results
# -----------------------------
display(uniform_df)          # Show enriched staff dataset
shipment_df.show(5)          # Show first 5 rows of shipment dataset

####Deriving of Columns (Time Intelligence)
Extracting temporal features from dates to enable period-based analysis and reporting.
Source File: logistics_shipment_detail_3000.json

#####1. Derive Shipment Year (shipment_year)

Scenario: Management needs an annual performance report to compare growth year-over-year.
Action: Extract the year component from shipment_date.
Result: "2024-04-23" -> 2024

#####2. Derive Shipment Month (shipment_month)

Scenario: Analysts want to identify seasonal peaks (e.g., increased volume in December).
Action: Extract the month component from shipment_date.
Result: "2024-04-23" -> 4 (April)

#####3. Flag Weekend Operations (is_weekend)

Scenario: The Operations team needs to track shipments handled during weekends to calculate overtime pay or analyze non-business day capacity.
Action: Flag as 'True' if the shipment_date falls on a Saturday or Sunday.

#####4. Flag shipment status (is_expedited)

Scenario: The Operations team needs to track shipments is IN_TRANSIT or DELIVERED.
Action: Flag as 'True' if the shipment_status IN_TRANSIT or DELIVERED.

In [0]:
# -----------------------------
# Add shipment year from shipment_date
# -----------------------------
shipment_df = shipment_df.withColumn("shipment_year", year(col("shipment_date")))

# Add shipment month from shipment_date
shipment_df = shipment_df.withColumn("shipment_month", month(col("shipment_date")))

# Add is_weekend flag (True if shipment_date is Saturday or Sunday)
shipment_df = shipment_df.withColumn(
    "is_weekend",
    when((dayofweek(col("shipment_date")) == 1) | (dayofweek(col("shipment_date")) == 7), True) # 1=Sunday, 7=Saturday
    .otherwise(False)
)

# Add shipment_status flag (mark expedited if status is IN_TRANSIT or DELIVERED)
shipment_df = shipment_df.withColumn(
    "is_expedited",
    when((col("shipment_status") == 'IN_TRANSIT') | (col("shipment_status") == 'DELIVERED'), True)
    .otherwise(False)
)

# Display shipment_date and is_weekend for inspection
display(shipment_df.select("shipment_date", "is_weekend"))

####Enrichment/Business Logics (Calculated Fields)
Deriving new metrics and financial indicators using mathematical and date-based operations.
Source File: logistics_shipment_detail_3000.json


#####1. Calculate Unit Cost (cost_per_kg)

Scenario: The Finance team wants to analyze the efficiency of shipments by determining the cost incurred per unit of weight.
Action: Divide shipment_cost by shipment_weight_kg.
Logic: shipment_cost / shipment_weight_kg

#####2. Track Shipment Age (days_since_shipment)

Scenario: The Operations team needs to monitor how long it has been since a shipment was dispatched to identify potential delays.
Action: Calculate the difference in days between the current_date and the shipment_date.
Logic: datediff(current_date(), shipment_date)

#####3. Compute Tax Liability (tax_amount)

Scenario: For invoicing and compliance, we must calculate the Goods and Services Tax (GST) applicable to each shipment.
Action: Calculate 18% GST on the total shipment_cost.
Logic: shipment_cost * 0.18

In [0]:
# -----------------------------
# Calculate unit cost per kg
# -----------------------------
shipment_df = shipment_df.withColumn(
    "cost_per_kg",
    try_divide(col("shipment_cost"), col("shipment_weight_kg"))  # Safe division, handles divide-by-zero gracefully
)

# -----------------------------
# Track shipment age in days
# -----------------------------
shipment_df = shipment_df.withColumn(
    "days_since_shipment",
    datediff(current_date(), col("shipment_date"))  # Difference between today and shipment_date
)

# -----------------------------
# Compute tax amount (18% of shipment_cost)
# -----------------------------
shipment_df = shipment_df.withColumn(
    "tax_amount",
    col("shipment_cost") * 0.18
)

# -----------------------------
# Display selected columns for inspection
# -----------------------------
display(
    shipment_df.select(
        "cost_per_kg",
        "shipment_cost",
        "shipment_weight_kg",
        "days_since_shipment",
        "shipment_date",
        "tax_amount",
        "shipment_cost"
    )
)

####Remove/Eliminate (drop, select, selectExpr)
Excluding unnecessary or redundant columns to optimize storage and privacy.
Source File: DF of logistics_source1 and logistics_source2


#####1. Remove Redundant Name Columns

- Scenario: Since we have already created the full_name column in the Enrichment step, the individual name columns are now redundant and clutter the dataset.
- Action: Drop the first_name and last_name columns.
- Logic: df.drop("first_name", "last_name")

In [0]:
# Drop unnecessary columns (staff_first_name and staff_last_name)
uniform_df = uniform_df.drop("staff_first_name", "staff_last_name")

# Select only the desired columns in a specific order
uniform_df = uniform_df.select(
    "shipment_id",      # Unique shipment identifier
    "full_name",        # Derived full name from first + last
    "age",              # Staff age
    "role",             # Staff role
    "origin_hub_city",  # Standardized hub location
    "vehicle_type",     # Standardized vehicle type
    "load_dt",          # Load timestamp
    "data_source"       # Source system or file identifier
)

# Display the cleaned DataFrame
display(uniform_df)

####Splitting & Merging/Melting of Columns
Reshaping columns to extract hidden values or combine fields for better analysis.
Source File: DF of logistics_shipment_detail_3000.json

- 1. Splitting (Extraction) Breaking one column into multiple to isolate key information.

Split Order Code:
 - Action: Split order_id ("ORD100000") into two new columns:
order_prefix ("ORD")
order_sequence ("100000")
- Split Date:
Action: Split shipment_date into three separate columns for partitioning:
ship_year (2024)
ship_month (4)
ship_day (23)

- 2. Merging (Concatenation) Combining multiple columns into a single unique identifier or description.

- - Create Route ID:
 - - Action: Merge source_city ("Chennai") and destination_city ("Pune") to create a descriptive route key:
route_lane ("Chennai->Pune")

In [0]:
# -----------------------------
# Split order_id into prefix and sequence
# -----------------------------
shipment_df = shipment_df \
    .withColumn("order_prefix", substring(col("order_id"), 1, 3)) \
    .withColumn("order_sequence", substring(col("order_id"), 4, length(col("order_id"))))
# Example: order_id = "ORD12345" → order_prefix = "ORD", order_sequence = "12345"

# -----------------------------
# Split shipment_date into year, month, day
# -----------------------------
shipment_df = shipment_df.withColumn("spilt_date", split(col("shipment_date"), "-"))

shipment_df = shipment_df \
    .withColumn("ship_year", col("spilt_date")[0]) \
    .withColumn("ship_month", col("spilt_date")[1]) \
    .withColumn("ship_day", col("spilt_date")[2]) \
    .drop("spilt_date")
# Example: shipment_date = "2025-12-15" → ship_year = "2025", ship_month = "12", ship_day = "15"

# -----------------------------
# Merge source and destination into route_lane
# -----------------------------
shipment_df = shipment_df.withColumn(
    "route_lane",
    concat(col("source_city"), lit("->"), col("destination_city"))
)
# Example: source_city = "Chennai", destination_city = "Delhi" → route_lane = "Chennai->Delhi"

#### 3. Data Customization & Processing - Application of Tailored Business Specific Rules
##### UDF1: Complex Incentive Calculation
#####Scenario: The Logistics Head wants to calculate a "Performance Bonus" for drivers based on tenure and role complexity.

Action: Create a Python function calculate_bonus(role, age) and register it as a Spark UDF.

Logic:

IF Role == 'Driver' AND Age > 50:
Bonus = 15% of Salary (Reward for Seniority)
IF Role == 'Driver' AND Age < 30:
Bonus = 5% of Salary (Encouragement for Juniors)
ELSE:
Bonus = 0
Result: A new derived column projected_bonus is generated for every row in the dataset.

UDF2: PII Masking (Privacy Compliance)
Scenario: For the analytics dashboard, we must hide the full identity of the staff to comply with privacy laws (GDPR/DPDP), while keeping names recognizable for internal managers.

Business Rule: Show the first 2 letters, mask the middle characters with ****, and show the last letter.

Action: Create a UDF mask_identity(name).

Example:

Input: "Rajesh"
Output: "Ra****h"
Note: Convert the above udf logic to inbult function based transformation to ensure the performance is improved.

In [0]:
# Define a Python function to calculate bonus based on role and age
def calculate_bonus(role: str, age: int):
    if role.upper() == "DRIVER" and age > 50:
        return 0.15   # Senior drivers get 15% bonus
    elif role.upper() == "DRIVER" and age < 30:
        return 0.05   # Young drivers get 5% bonus
    else:
        return 0      # Others get no bonus

# Register the function as a Spark UDF
bonus_udf = udf(calculate_bonus)

# Apply the UDF to the DataFrame, creating a new column 'projected_bonus'
uniform_df1 = uniform_df.withColumn("projected_bonus", bonus_udf(col("role"), col("age")))

# Display the enriched DataFrame
display(uniform_df1)

In [0]:
# Add projected_bonus column based on role and age
uniform_df_bonus = uniform_df.withColumn(
    "projected_bonus",
    when((col("role") == "driver") & (col("age") > 50), 0.15)   # Senior drivers get 15% bonus
    .when((col("role") == "driver") & (col("age") < 30), 0.05)  # Young drivers get 5% bonus
    .otherwise(0)                                               # Others get no bonus
)

# Display the enriched DataFrame
display(uniform_df_bonus)

In [0]:
# Define a Python function to mask identities
def mask_identities(name):
    if name is None:                      # If name is null, return None
        return None
    if len(name) <= 2:                    # If name length is 2 or less, return as-is
        return name
    # Otherwise, keep first 2 characters, last character, and mask the middle with '*'
    return name[:2] + "*" * (len(name) - 3) + name[-1]

# Register the function as a Spark UDF
mask_identities_udf = udf(mask_identities)

# Check available columns in uniform_df
uniform_df.columns

# Apply the UDF to mask the 'full_name' column
uniform_df1 = uniform_df.withColumn("full_name", mask_identities_udf(col("full_name")))

# Display the masked DataFrame
display(uniform_df1)

In [0]:
# Mask the full_name column using native Spark functions
uniform_df = uniform_df.withColumn(
    "full_name",
    concat(
        substring(col("full_name"), 1, 2),                        # Keep the first 2 characters
        repeat(lit("*"), length(col("full_name")) - 3),           # Replace middle characters with '*'
        substring(col("full_name"), -1, 1)                        # Keep the last character
    )
)

# Display the masked DataFrame
display(uniform_df)

####4. Data Core Curation & Processing (Pre-Wrangling)
Applying business logic to focus, filter, and summarize data before final analysis.


#####1. Select (Projection)
Source Files: DF of logistics_source1 and logistics_source2

Scenario: The Driver App team only needs location data, not sensitive HR info.
Action: Select only first_name, role, and hub_location.

#####2. Filter (Selection)
Source File: DF of json

Scenario: We need a report on active operational problems.
Action: Filter rows where shipment_status is 'DELAYED' or 'RETURNED'.
Scenario: Insurance audit for senior staff.
Action: Filter rows where age > 50.

#####3. Derive Flags & Columns (Business Logic)
Source File: DF of json

Scenario: Identify high-value shipments for security tracking.
Action: Create flag is_high_value = True if shipment_cost > 50,000.
Scenario: Flag weekend operations for overtime calculation.
Action: Create flag is_weekend = True if day is Saturday or Sunday.

#####4. Format (Standardization)
Source File: DF of json

Scenario: Finance requires readable currency formats.
Action: Format shipment_cost to string like "₹30,695.80".
Scenario: Standardize city names for reporting.
Action: Format source_city to Uppercase (e.g., "chennai" → "CHENNAI").

#####5. Group & Aggregate (Summarization)
Source Files: DF of logistics_source1 and logistics_source2

Scenario: Regional staffing analysis.
Action: Group by hub_location and Count the number of staff.
Scenario: Fleet capacity analysis.
Action: Group by vehicle_type and Sum the shipment_weight_kg.

#####6. Sorting (Ordering)
Source File: DF of json

Scenario: Prioritize the most expensive shipments.
Action: Sort by shipment_cost in Descending order.
Scenario: Organize daily dispatch schedule.
Action: Sort by shipment_date (Ascending) then priority_flag (Descending).

#####7. Limit (Top-N Analysis)
Source File: DF of json

Scenario: Dashboard snapshot of critical delays.
Action: Filter for 'DELAYED', Sort by Cost, and Limit to top 10 rows.

In [0]:
# ----------------------------------------
# Selecting required columns for Driver Team
# ----------------------------------------
# Keep only full_name, role, and origin_hub_city from uniform_df
driver_team_df = uniform_df.select("full_name", "role", "origin_hub_city")

# ----------------------------------------
# Filtering rows based on conditions
# ----------------------------------------
# Operational problems: shipments with status DELAYED or RETURNED
operational_prblm_df = shipment_df.where(col("shipment_status").isin(["DELAYED", "RETURNED"]))

# Insurance audit: staff older than 50 years
audit_df = uniform_df.where(col("age") > 50)

# ----------------------------------------
# Deriving flags (business logic)
# ----------------------------------------
# Flag shipments as high value if shipment_cost > 50,000
shipment_df = shipment_df.withColumn(
    "is_high_value",
    when(col("shipment_cost") > 50000, lit(True)).otherwise(lit(False))
)

# Uncomment below line to preview the flag with shipment_cost
# display(shipment_df.select("is_high_value", "shipment_cost"))

# ----------------------------------------
# Format Standardization
# ----------------------------------------
# Convert shipment_cost to string for finance reporting
shipment_df = shipment_df.withColumn("finance_shipment_cost", col("shipment_cost").cast("string"))

# Add currency symbol "$" to shipment_cost for readability
shipment_df = shipment_df.withColumn("finance_shipment_cost", concat(lit("$"), col("shipment_cost")))

# Standardize city names to uppercase for consistency in reporting
shipment_df = shipment_df.withColumn("source_city", upper(col("source_city")))

In [0]:
# ----------------------------------------
# Grouping staff data by hub location
# ----------------------------------------
# Count the number of staff members in each origin hub city
display(
    uniform_df.groupBy("origin_hub_city")
              .agg(count("*").alias("Hub_wise_staff_count"))
)

# ----------------------------------------
# Grouping shipment data by vehicle type
# ----------------------------------------
# Sum the shipment weight (in kg) for each vehicle type
display(
    shipment_df.groupBy("vehicle_type")
               .agg(sum("shipment_weight_kg").alias("Total_weight_kg"))
)

In [0]:
# ----------------------------------------
# Sorting shipments by cost (descending)
# ----------------------------------------
# This will prioritize the most expensive shipments first
display(
    shipment_df.orderBy("shipment_cost", ascending=False)
)

# ----------------------------------------
# Sorting shipments by date and high-value flag
# ----------------------------------------
# First sort by shipment_date (ascending → earliest first),
# then sort by is_high_value (descending → True values first)
display(
    shipment_df.orderBy("shipment_date", "is_high_value", ascending=[True, False])
)

In [0]:
# ----------------------------------------
# Top-N Analysis: Critical Delays
# ----------------------------------------
# Step 1: Filter shipments where status is 'DELAYED'
# Step 2: Sort these delayed shipments by shipment_cost in descending order
# Step 3: Limit the result to the top 10 most expensive delayed shipments
display(
    shipment_df.where(col("shipment_status") == "DELAYED")
               .orderBy(col("shipment_cost").desc())
               .limit(10)
)

####5. Data Wrangling - Transformation & Analytics
Combining, modeling, and analyzing data to answer complex business questions.

######1. Joins
Source Files:
Left Side (staff_df):
DF of logistics_source1 & logistics_source2
Right Side (shipments_df):
DF of logistics_shipment_detail_3000.json

######1.1 Frequently Used Simple Joins (Inner, Left)
Inner Join (Performance Analysis):
Scenario: We only want to analyze completed work. Connect Staff to the Shipments they handled.
Action: Join staff_df and shipments_df on shipment_id.
Result: Returns only rows where a staff member is assigned to a valid shipment.
Left Join (Idle Resource check):
Scenario: Find out which staff members are currently idle (not assigned to any shipment).
Action: Join staff_df (Left) with shipments_df (Right) on shipment_id. Filter where shipments_df.shipment_id is NULL.

######1.2 Infrequent Simple Joins (Self, Right, Full, Cartesian)
Self Join (Peer Finding):
Scenario: Find all pairs of employees working in the same hub_location.
Action: Join staff_df to itself on hub_location, filtering where staff_id_A != staff_id_B.
Right Join (Orphan Data Check):
Scenario: Identify shipments in the system that have no valid driver assigned (Data Integrity Issue).
Action: Join staff_df (Left) with shipments_df (Right). Focus on NULLs on the left side.
Full Outer Join (Reconciliation):
Scenario: A complete audit to find both idle drivers AND unassigned shipments in one view.
Action: Perform a Full Outer Join on shipment_id.
Cartesian/Cross Join (Capacity Planning):
Scenario: Generate a schedule of every possible driver assignment to every pending shipment to run an optimization algorithm.
Action: Cross Join drivers_df and pending_shipments_df.

######1.3 Advanced Joins (Semi and Anti)
Left Semi Join (Existence Check):
Scenario: "Show me the details of Drivers who have at least one shipment." (Standard filtering).
Action: staff_df.join(shipments_df, "shipment_id", "left_semi").
Benefit: Performance optimization; it stops scanning the right table once a match is found.
Left Anti Join (Negation Check):
Scenario: "Show me the details of Drivers who have never touched a shipment."
Action: staff_df.join(shipments_df, "shipment_id", "left_anti").

####2. Lookup
Source File: DF of logistics_source1 and logistics_source2 (merged into Staff DF)

Scenario: Validation. Check if the hub_location in the staff file exists in the corporate Master_City_List.
Action: Compare values against a reference list.

#####3. Lookup & Enrichment
Source File: DF of logistics_source1 and logistics_source2 (merged into Staff DF)

Scenario: Geo-Tagging.
Action: Lookup hub_location ("Pune") in a Master Latitude/Longitude table and enrich the dataset by adding lat and long columns for map plotting.

####4. Schema Modeling (Denormalization)
Source Files: DF of All 3 Files (logistics_source1, logistics_source2, logistics_shipment_detail_3000.json)

Scenario: Creating a "Gold Layer" Table for PowerBI/Tableau.
Action: Flatten the Star Schema. Join Staff, Shipments, and Vehicle_Master into one wide table (wide_shipment_history) so analysts don't have to perform joins during reporting.

#####5. Windowing (Ranking & Trends)
Source Files:
DF of logistics_source2: Provides hub_location (Partition Key).
logistics_shipment_detail_3000.json: Provides shipment_cost (Ordering Key)

Scenario: "Who are the Top 3 Drivers by Cost in each Hub?"
Action:
Partition by hub_location.
Order by total_shipment_cost Descending.
Apply dense_rank() and `row_number()
Filter where rank or row_number <= 3.

####6. Analytical Functions (Lead/Lag)
Source File:
DF of logistics_shipment_detail_3000.json

Scenario: Idle Time Analysis.
Action: For each driver, calculate the days elapsed since their previous shipment.

####7. Set Operations
Source Files: DF of logistics_source1 and logistics_source2

Union: Combining Source1 (Legacy) and Source2 (Modern) into one dataset (Already done in Active Munging).
Intersect: Identifying Staff IDs that appear in both Source 1 and Source 2 (Duplicate/Migration Check).
Except (Difference): Identifying Staff IDs present in Source 2 but missing from Source 1 (New Hires).

####8. Grouping & Aggregations (Advanced)
Source Files:
DF of logistics_source2: Provides hub_location and vehicle_type (Grouping Dimensions).
DF of logistics_shipment_detail_3000.json: Provides shipment_cost (Aggregation Metric).

Scenario: The CFO wants a subtotal report at multiple levels:
Total Cost by Hub.
Total Cost by Hub AND Vehicle Type.
Grand Total.
Action: Use cube("hub_location", "vehicle_type") or rollup() to generate all these subtotals in a single query.

In [0]:
# ----------------------------------------
# Basic Joins (Inner and Left)
# ----------------------------------------

# Inner Join
# Issue: Both DataFrames have a column named 'vehicle_type'.
# To avoid duplicate column name conflicts, rename in shipment_df.
shipment_df = shipment_df.withColumnRenamed("vehicle_type", "shipment_vehicle_type")

# Perform inner join on shipment_id
inner_joined_df = uniform_df.join(shipment_df, how="inner", on="shipment_id")

# Example: You can filter joined results further, e.g. only delivered shipments
# display(uniform_df.join(shipment_df, how="inner", on="shipment_id").where(col("shipment_status") == "DELIVERED"))

# ----------------------------------------
# Left Join
# ----------------------------------------
# Alias DataFrames for clarity
u = uniform_df.alias("u")
s = shipment_df.alias("s")

# Perform left join on shipment_id
# Then filter rows where shipment_id from shipment_df is NULL (i.e., no match found)
# Finally, select only columns from uniform_df
display(
    u.join(s, how="left", on=col("u.shipment_id") == col("s.shipment_id"))
     .where(col("s.shipment_id").isNull())
     .select("u.*")
)

In [0]:
# ----------------------------------------
# Self Join on uniform_df
# ----------------------------------------
# Goal: Find pairs of shipments that originate from the same hub city
# but have different shipment_ids.

display(
    uniform_df.alias("a")   # First alias of uniform_df
    .join(
        uniform_df.alias("b"),                          # Second alias of uniform_df
        on=col("a.origin_hub_city") == col("b.origin_hub_city")  # Join condition: same hub city
    )
    .where(col("a.shipment_id") != col("b.shipment_id")) # Exclude identical shipment_id matches
    .select(
        "a.shipment_id", "a.origin_hub_city",           # Select shipment_id + hub city from alias a
        "b.shipment_id", "b.origin_hub_city"            # Select shipment_id + hub city from alias b
    )
)

In [0]:
# ----------------------------------------
# Right Join
# ----------------------------------------
# Join uniform_df (u) with shipment_df (s) on shipment_id
# Scenario 1: Find shipments that have no matching staff record
display(
    uniform_df.alias("u")
    .join(shipment_df.alias("s"), how="right", on=col("u.shipment_id") == col("s.shipment_id"))
    .where(col("u.shipment_id").isNull())   # staff record missing
    .select("s.*")                          # show shipment details only
)

# Scenario 2: Check if shipment is assigned to a driver
display(
    uniform_df.alias("u")
    .join(shipment_df.alias("s"), how="right", on=col("u.shipment_id") == col("s.shipment_id"))
    .where((col("u.shipment_id").isNotNull()) & (col("u.role") == "driver"))  # staff exists and role is driver
    .select("u.*")                                                              # show driver details
)

# ----------------------------------------
# Full Outer Join
# ----------------------------------------
# Capture mismatches: records present in one DF but not the other
display(
    uniform_df.alias("u")
    .join(shipment_df.alias("s"), col("u.shipment_id") == col("s.shipment_id"), how="full")
    .where((col("u.shipment_id").isNull()) | (col("s.shipment_id").isNull()))   # either side missing
)

# ----------------------------------------
# Cross Join
# ----------------------------------------
# Scenario: Match all drivers with all pending shipments
pending_shipment_df = shipment_df.where((col("shipment_status") == "IN_TRANSIT") | (col("shipment_status") == "DELAYED"))
driver_df = uniform_df.where(col("role") == "driver")

# Cross join creates Cartesian product: every driver paired with every pending shipment
# Useful for assignment simulations or matching logic
# display(driver_df.join(pending_shipment_df, how="cross"))

In [0]:
# ----------------------------------------
# Left Semi Join
# ----------------------------------------
# Keeps only rows from uniform_df where shipment_id exists in shipment_df.
# Equivalent to filtering uniform_df for matching shipment_ids.
display(
    uniform_df.join(shipment_df, how="left_semi", on="shipment_id")
)

# ----------------------------------------
# Left Anti Join
# ----------------------------------------
# Keeps only rows from uniform_df where shipment_id does NOT exist in shipment_df.
# Equivalent to filtering uniform_df for non-matching shipment_ids.
display(
    uniform_df.join(shipment_df, how="left_anti", on="shipment_id")
)

In [0]:
# ----------------------------------------
# Load Master City List
# ----------------------------------------
# Read the master city list CSV file into a DataFrame
# header=True → first row contains column names
# inferSchema=True → automatically detect column types
master_city_list = spark.read.csv(
    "dbfs:/Volumes/usecase_data/logistics_proj_data/projdata/Master_City_List.csv",
    header=True,
    inferSchema=True
)

# ----------------------------------------
# Lookup: Validate hub cities against master list
# ----------------------------------------
# Left semi join → keeps only rows from uniform_df where origin_hub_city exists in master_city_list
# Then select distinct hub cities to see which ones are valid
display(
    uniform_df.alias("u")
    .join(master_city_list.alias("m"), how="left_semi", on=col("u.origin_hub_city") == col("m.city_name"))
    .select("u.origin_hub_city")
    .distinct()
)

# ----------------------------------------
# Lookup + Enrichment: Add geo details to staff records
# ----------------------------------------
# Left join → keeps all rows from uniform_df and enriches with matching city info from master_city_list
geo_tagging_df = (
    uniform_df.alias("u")
    .join(master_city_list.alias("m"), how="left", on=col("u.origin_hub_city") == col("m.city_name"))
)

# Example: geo_tagging_df will now contain columns from both uniform_df and master_city_list
# allowing you to enrich staff records with city metadata (e.g., region, state, coordinates)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank, row_number

# ----------------------------------------
# Schema Modeling (Denormalisation)
# ----------------------------------------
# Join uniform_df (staff data) with shipment_df (shipment data) on shipment_id
# This creates a denormalized DataFrame combining staff and shipment details
joined_df = uniform_df.join(shipment_df, how="inner", on="shipment_id")

# ----------------------------------------
# Windowing Functions
# ----------------------------------------
# Define a window specification:
# Partition by origin_hub_city (group by hub city)
# Order shipments within each hub by shipment_cost in descending order
window_spec = Window.partitionBy("origin_hub_city").orderBy(col("shipment_cost").desc())

# ----------------------------------------
# Dense Rank Example
# ----------------------------------------
# Assign a dense rank to shipments within each hub based on cost
# Dense rank ensures ties get the same rank, and next rank is incremented
# Filter to keep only top 3 shipments per hub
display(
    joined_df.withColumn("dense_rk", dense_rank().over(window_spec))
             .where(col("dense_rk") <= 3)
)

# ----------------------------------------
# Row Number Example
# ----------------------------------------
# Assign a unique row number to shipments within each hub based on cost
# Row number increments sequentially without gaps
# Filter to keep only top 3 shipments per hub
top3_driver_df = (
    joined_df.withColumn("row_num", row_number().over(window_spec))
             .where(col("row_num") <= 3)
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, datediff

# ----------------------------------------
# Display shipment DataFrame
# ----------------------------------------
display(shipment_df)

# ----------------------------------------
# Window Specification
# ----------------------------------------
# Partition by shipment_id → analyze each shipment separately
# Order by shipment_date → chronological order of events for each shipment
window_spec1 = Window.partitionBy(col("shipment_id")).orderBy(col("shipment_date"))

# ----------------------------------------
# Idle Driver Analysis
# ----------------------------------------
# Use lag() to get the previous shipment_date for each shipment_id
# Calculate idle_time as the difference (in days) between current and previous shipment_date
idle_driver_analysis = shipment_df.withColumn(
    "idle_time",
    datediff(col("shipment_date"), lag(col("shipment_date"), 1).over(window_spec1))
)

# Display the results with idle_time column
display(idle_driver_analysis)

In [0]:
from pyspark.sql.functions import lit

# ----------------------------------------
# Schema Alignment for Set Operations
# ----------------------------------------
# Union, Intersect, and Except require both DataFrames to have the same number of columns.
# To align schemas, add missing columns with NULL values to active_mung_df1.
active_mung_df3 = (
    active_mung_df1.withColumn("hub_location", lit(None))
                   .withColumn("vehicle_type", lit(None))
)

# ----------------------------------------
# Union
# ----------------------------------------
# Union joins by position → column order and types must match.
# Union includes duplicates.
# This fails if column counts differ (e.g., 6 vs 8 columns).
All_staff_data = active_mung_df3.union(active_mung_df2)

# ----------------------------------------
# Union By Name
# ----------------------------------------
# Safer alternative: matches columns by name, not position.
# allowMissingColumns=True → fills missing columns with NULLs.
active_mung_df3.unionByName(active_mung_df2, allowMissingColumns=True)

# ----------------------------------------
# Intersect
# ----------------------------------------
# Returns rows present in both DataFrames (no duplicates).
# Requires same number of columns.
duplicate_staff_id = active_mung_df3.intersect(active_mung_df2)

# ----------------------------------------
# ExceptAll
# ----------------------------------------
# Returns rows in df1 but not in df2, keeping duplicates.
New_hires_list = active_mung_df3.exceptAll(active_mung_df2)

# ----------------------------------------
# Except (without duplicates)
# ----------------------------------------
# Similar to ExceptAll but removes duplicates.
# Uncomment below if needed:
# active_mung_df1.except(active_mung_df2).display()

In [0]:
from pyspark.sql.functions import sum

# ----------------------------------------
# Multi-level Subtotal Report using Cube
# ----------------------------------------
# Cube generates subtotals for all combinations of the specified dimensions.
# Dimensions: origin_hub_city, vehicle_type
# Aggregation: sum of shipment_cost
multilevel_subtotal_report = (
    inner_joined_df
    .cube("origin_hub_city", "vehicle_type")
    .agg(sum("shipment_cost").alias("total_cost"))
)

# Display the cube results
display(multilevel_subtotal_report)

###6. Data Persistance (LOAD)-> Data Publishing & Consumption
Store the inner joined, lookup and enrichment, Schema Modeling, windowing, analytical functions, set operations, grouping and aggregation data into the delta tables.

In [0]:
# ----------------------------------------
# Persisting DataFrames as Managed Tables
# ----------------------------------------
# mode="overwrite" → replaces existing table if it already exists

# Denormalized staff + shipment data
inner_joined_df.write.saveAsTable(
    "usecase_data.logistics_proj_data.denormalized_table", mode="overwrite"
)

# Geo-tagging enriched staff data
geo_tagging_df.write.saveAsTable(
    "usecase_data.logistics_proj_data.geo_tag_table", mode="overwrite"
)

# Wide shipment history (joined view of staff + shipment)
inner_joined_df.write.saveAsTable(
    "usecase_data.logistics_proj_data.wide_shipment_history", mode="overwrite"
)

# Top 3 performing drivers per hub (windowing result)
top3_driver_df.write.saveAsTable(
    "usecase_data.logistics_proj_data.top_performing_driver", mode="overwrite"
)

# Idle driver analysis (lag-based idle time calculation)
idle_driver_analysis.write.saveAsTable(
    "usecase_data.logistics_proj_data.idle_driver_analysis", mode="overwrite"
)

# Consolidated staff data (union result)
All_staff_data.write.saveAsTable(
    "usecase_data.logistics_proj_data.all_staff_data", mode="overwrite"
)

# New hires list (exceptAll result)
New_hires_list.write.saveAsTable(
    "usecase_data.logistics_proj_data.new_hires_list", mode="overwrite"
)

# Duplicate staff IDs (intersect result)
duplicate_staff_id.write.saveAsTable(
    "usecase_data.logistics_proj_data.duplicate_staff_list", mode="overwrite"
)

# Multi-level subtotal report (cube aggregation)
multilevel_subtotal_report.write.saveAsTable(
    "usecase_data.logistics_proj_data.multilevel_subtotal_report", mode="overwrite"
)